In [ ]:
from dotenv import load_dotenv

from langchain_teddynote import logging
from langchain.storage import LocalFileStore, InMemoryByteStore

from langchain_openai import OpenAIEmbeddings
from langchain.embeddings import CacheBackedEmbeddings

from langchain.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores.faiss import FAISS

In [ ]:
load_dotenv()

In [ ]:
logging.langsmith("langchain-08")

In [ ]:
embedding = OpenAIEmbeddings()

LocalFileStore로 임베딩을 영구적으로 저장

In [ ]:
store1 = LocalFileStore("./cache/")  # 로컬 파일 저장소

cached_embedder1 = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding, 
    document_embedding_cache=store1, 
    namespace=embedding.model  # 기본 임베딩과 저장소를 사용하여 캐시 지원 임베딩을 생성; 임베딩 값 구분자
)

In [ ]:
list(store.yield_keys())  # store에서 키들을 순차적으로 가져오기

In [ ]:
raw_documents = TextLoader("./data/appendix-keywords.txt").load()
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
documents = text_splitter.split_documents(raw_documents)

In [ ]:
%time db = FAISS.from_documents(documents, cached_embedder1)  # FAISS DB 생성 + 실행 시간 측정

In [ ]:
%time db2 = FAISS.from_documents(documents, cached_embedder1)  # 이렇게 캐싱을 이용하면 임베딩 다시 계산할 필요 X

InmemoryByteStore로 임베딩을 비영구적으로 저장

In [ ]:
store2 = InMemoryByteStore()

cached_embedder2 = CacheBackedEmbeddings.from_bytes_store(
    underlying_embeddings=embedding, 
    document_embedding_cache=store2, 
    namespace=embedding.model
)